# W02 · 02 DOI 정규화와 중복 제거 (10분)

**무엇** — `norm_doi()` 를 쓰고, 같은 267편이 든 서지 파일 2개를 합쳐(534행) 중복을 제거해 267편을 확인한다.
**왜** — 수집 폴더마다 DOI 표기가 달라(URL 접두 · 대문자 · 공백) 그대로 세면 편수가 부풀려진다. 슬라이드 24 의 라이브와 같은 규칙.

실행(week02 폴더 · Claude Code): 「@notebooks/02_dedup_doi.ipynb 를 실행하고 결과를 요약해」 → 승인 창의 `python ../run_nb.py notebooks/02_dedup_doi.ipynb` 확인 · 승인 → `out/dedup_report.json` 을 `expected_output/` 과 비교. 오류가 나면 덱 12장 P2-2 지시를 쓴다.

규칙 출처: 연구 플랫폼 `scripts/build_corpus_index.py` 의 `norm_doi()` — strip·끝 마침표 제거 → URL·doi: 접두 제거 → 소문자.

In [ ]:
import re, json, pathlib
import pandas as pd

def norm_doi(d):
    """DOI 표기 변형 → 하나의 키. 빈 값이면 ''."""
    if not isinstance(d, str):
        return ''
    d = d.strip().rstrip('.')                                    # ① 공백 · 끝 마침표
    d = re.sub(r'^https?://(dx\.)?doi\.org/', '', d, flags=re.I) # ② URL 접두
    d = re.sub(r'^doi:\s*', '', d, flags=re.I)                   # ② doi: 접두
    return d.lower()                                             # ③ 소문자

# 슬라이드 24 의 8건 예시로 먼저 확인
raw = ['10.1016/j.conbuildmat.2026.146313', 'https://doi.org/10.1016/j.conbuildmat.2026.146313', '10.1016/J.CONBUILDMAT.2026.146313',
       '10.1016/j.fuel.2024.133011', 'doi: 10.1016/j.fuel.2024.133011', ' 10.1016/j.fuel.2024.133011. ',
       '10.1016/j.molliq.2025.127679', 'https://dx.doi.org/10.1016/J.MOLLIQ.2025.127679']
keys = [norm_doi(d) for d in raw]
print('레코드', len(raw), '→ 고유 키', len(set(keys)))
for k in sorted(set(keys)): print(' ', k)

In [ ]:
# 두 서지 파일 합치기 — 같은 267편 (md_corpus_267 과 md_asphalt_literature 는 DOI 집합이 같다)
YEAR_MIN = 2021   # 바꿔 보기(덱 13장): 2023 으로 바꿔 다시 실행 → unique 177 · 기본 2021 = 자료 전체(2021~2026)
a = pd.read_csv('data/md_corpus_267.csv')
b = pd.read_csv('data/md_asphalt_literature_2021_2026.csv')
df = pd.concat([a, b], ignore_index=True)
df = df[df['year'] >= YEAR_MIN].reset_index(drop=True)
print('합친 행 수:', len(df))
df['key'] = df['doi'].map(norm_doi)
print('정규화 전 고유:', df['doi'].nunique(), '· 정규화 후 고유:', df['key'].nunique())

In [ ]:
# 연습: 일부러 표기를 흐트러뜨린 뒤 다시 세어 본다 — 정규화가 없으면 고유 수가 부풀려진다
messy = df.copy()
messy.loc[::3, 'doi'] = 'https://doi.org/' + messy.loc[::3, 'doi']   # 3행마다 URL 접두
messy.loc[1::3, 'doi'] = messy.loc[1::3, 'doi'].str.upper()           # 대문자
print('흐트러진 뒤 정규화 전 고유:', messy['doi'].nunique())
print('정규화 후 고유:', messy['doi'].map(norm_doi).nunique(), '(267 이어야 한다)')

In [ ]:
# 보고서 저장 — 이 파일 하나를 제출한다
from collections import Counter
counts = Counter(k for k in df['key'] if k)
report = {'inputs': ['md_corpus_267.csv', 'md_asphalt_literature_2021_2026.csv'], 'records': int(len(df)),
          'unique_raw': int(df['doi'].nunique()), 'unique': int(len(counts)), 'duplicates': int(len(df) - len(counts)),
          'missing_doi': int((df['key'] == '').sum()), 'top_duplicated': counts.most_common(3),
          'rule': "strip·rstrip('.') → ^https?://(dx.)?doi.org/ 제거 → ^doi: 제거 → lower()"}
pathlib.Path('out').mkdir(exist_ok=True)
pathlib.Path('out/dedup_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))
print('\n완료:', 'unique = 267' if report['unique'] == 267 else '값 확인 필요')

### 해 보기
1. 단계 ③(소문자)만 빼고 다시 세어 본다 — 고유 수가 얼마나 늘어나나?
2. `comid_unified_bibliography.csv`(530행)를 같은 방법으로 세어 본다. 이 파일은 DOI 중복이 없어야 한다.
3. DOI 가 없는 행은 어떤 키로 합쳐야 할까? (플랫폼 규칙: PII → 제목 앞 60자)